# Copilot Phase 0 — Reconciled Loan Portfolio KPIs

Build one record per loan and observed payment period, then reconcile portfolio and segment KPIs to that grain. Periods are indexed by the loan's payment sequence. The most common due-date month is a display label.

Set `LOAN_DATA_DIR` to the original competition CSV directory. By default the notebook writes the aggregate outputs to the current directory. Set `LOAN_KPI_OUTPUT_DIR` to use another output directory.

## 0 · Imports & Config

In [ ]:
import os
from pathlib import Path

import pandas as pd

DATA_DIR = Path(os.environ.get("LOAN_DATA_DIR", "."))
OUTPUT_DIR = Path(os.environ.get("LOAN_KPI_OUTPUT_DIR", "."))
PAYMENT_KEY = ["loan_id", "month_number"]

## 1 · Load Raw Tables

In [ ]:
raw_payments = pd.read_csv(DATA_DIR / "payment_history.csv")
raw_loans = pd.read_csv(DATA_DIR / "loans.csv")
raw_borrowers = pd.read_csv(DATA_DIR / "borrowers.csv")

for table, key in (
    (raw_payments, PAYMENT_KEY),
    (raw_loans, ["loan_id"]),
    (raw_borrowers, ["borrower_id"]),
):
    if table[key].isna().any().any():
        raise ValueError(f"Missing key in source columns {key}.")

payment_history = raw_payments.drop_duplicates()
loans = raw_loans.drop_duplicates()
borrowers = raw_borrowers.drop_duplicates()

if payment_history.empty or payment_history.duplicated(PAYMENT_KEY).any():
    raise ValueError("Expected one payment row per loan and period after removing exact duplicates.")
if borrowers.duplicated("borrower_id").any():
    raise ValueError("Conflicting records for the same borrower.")
if loans.drop(columns="borrower_id").drop_duplicates().duplicated("loan_id").any():
    raise ValueError("Conflicting loan attributes for the same loan.")

# Keep the agreed loan attributes while withholding an ambiguous borrower assignment.
ambiguous_ids = loans.loc[loans.duplicated("loan_id", keep=False), "loan_id"].unique()
loans = loans.drop_duplicates("loan_id").copy()
loans.loc[loans["loan_id"].isin(ambiguous_ids), "borrower_id"] = pd.NA

if not payment_history["loan_id"].isin(loans["loan_id"]).all():
    raise ValueError("Payment history contains loan IDs missing from the loan table.")
period = payment_history["month_number"]
if not (period.ge(1) & period.mod(1).eq(0)).all():
    raise ValueError("Payment periods must be positive integers.")
if payment_history["days_delinquent"].lt(0).any():
    raise ValueError("Days delinquent must be nonnegative when recorded.")

### 1.1 Schema & null audit

In [ ]:
source_audit = pd.DataFrame([
    ("borrowers", len(raw_borrowers), len(borrowers), 0),
    ("loans", len(raw_loans), len(loans), len(ambiguous_ids)),
    ("loan-periods", len(raw_payments), len(payment_history), 0),
], columns=["source", "raw_rows", "unique_keys", "ambiguous_borrower_assignments"])
source_audit

---
## Step 1 · Loan-Level Cumulative Snapshot

For every `(loan_id, month_number)` pair we compute **expanding-window** aggregates 
— i.e. metrics that cover **month 1 through the current month** inclusive.

| Column | Definition |
|---|---|
| `on_time_flag` | 1 if `days_delinquent == 0` (or null, treated as 0), else 0 |
| `on_time_rate_cumul` | cumulative on-time payments / cumulative total payments |
| `total_delinquent_cumul` | cumulative sum of `days_delinquent` up to this month |
| `months_observed` | how many months of history exist up to this point |
| `months_delinquent_cumul` | count of months where `days_delinquent > 0` |
| `worst_delinquent_month` | worst single-month `days_delinquent` seen so far |

### 1.1 Derive `on_time_flag` from `days_delinquent`

In [ ]:
ph = payment_history.sort_values(PAYMENT_KEY).copy()
ph["status_known"] = ph["days_delinquent"].notna().astype(int)
ph["on_time"] = ph["days_delinquent"].eq(0).astype(int)
ph["delinquent"] = ph["days_delinquent"].gt(0).astype(int)
ph["recorded_delinquent_days"] = ph["days_delinquent"].fillna(0)

### 1.2 Expanding-window cumulative metrics

We use `groupby + expanding()` so that for loan L001 at month 3 we only look at months 1–3, 
not months 4–7.  This faithfully mimics a real-time monitoring system.

In [ ]:
by_loan = ph.groupby("loan_id", sort=False)
ph["months_with_known_status"] = by_loan["status_known"].cumsum()
ph["on_time_count_cumul"] = by_loan["on_time"].cumsum()
ph["months_delinquent_cumul"] = by_loan["delinquent"].cumsum()
ph["total_delinquent_cumul"] = by_loan["recorded_delinquent_days"].cumsum()
denominator = ph["months_with_known_status"].where(ph["months_with_known_status"].gt(0))
ph["on_time_rate_cumul"] = ph["on_time_count_cumul"] / denominator

### 1.3 Month-over-month change in `on_time_rate_cumul`

Captures whether a loan is **deteriorating** — its cumulative on-time rate is falling. 
This feeds the *new deteriorating loans* KPI in Step 2.

In [ ]:
DETERIORATION_THRESHOLD = -0.10
rate_change = by_loan["on_time_rate_cumul"].diff()
comparable = (
    by_loan["month_number"].diff().eq(1)
    & ph["status_known"].eq(1)
    & rate_change.notna()
)
ph["on_time_rate_mom_change"] = rate_change.where(comparable)
ph["is_deteriorating"] = (
    rate_change.lt(DETERIORATION_THRESHOLD - 1e-12).astype(float).where(comparable)
)

### 1.4 Attach loan metadata

Join `loan_type` and `state` (via borrowers) so downstream groupings work.

In [ ]:
loan_meta = loans[["loan_id", "borrower_id", "loan_type", "principal"]].merge(
    borrowers[["borrower_id", "state"]], on="borrower_id", how="left", validate="many_to_one"
)
loan_monthly_snapshot = ph.merge(loan_meta, on="loan_id", how="left", validate="many_to_one")

### 1.5 Reconcile the unchecked joins to the source

In [ ]:
# Reproduce the row multiplication of the prior unchecked joins on the same payment keys.
borrower_matches = raw_loans["borrower_id"].map(raw_borrowers["borrower_id"].value_counts()).fillna(1)
raw_loan_matches = raw_loans["loan_id"].value_counts()
raw_dimension_matches = borrower_matches.groupby(raw_loans["loan_id"]).sum()
reconciliation = payment_history[PAYMENT_KEY].assign(
    rows_after_unchecked_loan_join=lambda frame: frame["loan_id"].map(raw_loan_matches),
    rows_after_unchecked_dimension_joins=lambda frame: frame["loan_id"].map(raw_dimension_matches),
).groupby("month_number").agg(
    source_loan_periods=("loan_id", "size"),
    rows_after_unchecked_loan_join=("rows_after_unchecked_loan_join", "sum"),
    rows_after_unchecked_dimension_joins=("rows_after_unchecked_dimension_joins", "sum"),
).astype(int)
reconciliation["extra_rows_from_unchecked_joins"] = (
    reconciliation["rows_after_unchecked_dimension_joins"] - reconciliation["source_loan_periods"]
)
reconciliation

### 1.6 Sanity checks

Validate the expanding-window logic on a few example loans.

In [ ]:
source_keys = pd.MultiIndex.from_frame(payment_history[PAYMENT_KEY]).sort_values()
snapshot_keys = pd.MultiIndex.from_frame(loan_monthly_snapshot[PAYMENT_KEY]).sort_values()
assert source_keys.equals(snapshot_keys), "Snapshot must preserve every unique source payment key."
assert loan_monthly_snapshot["on_time_rate_cumul"].dropna().between(0, 1).all()
assert by_loan["total_delinquent_cumul"].diff().dropna().ge(0).all()
print(f"Reconciled {len(snapshot_keys):,} loan-periods.")

### 1.7 Distribution preview — cumulative KPIs at final observed month

For each loan, look at the KPIs as of its **last recorded month** — 
this represents the most current risk reading per loan.

In [ ]:
latest = loan_monthly_snapshot.drop_duplicates("loan_id", keep="last")
latest[["on_time_rate_cumul", "total_delinquent_cumul"]].describe()

---
## ✓ Step 1 Complete

The `loan_monthly_snapshot` DataFrame is ready:  
**{rows:,} rows × {cols} cols** — one row per loan per month, 
with expanding-window cumulative KPIs.

**Next:** Step 2 — aggregate to portfolio-level monthly KPIs → `portfolio_monthly_kpi`.

In [ ]:
source_audit.assign(rows_removed=source_audit["raw_rows"] - source_audit["unique_keys"])

---
## Step 2 · Portfolio-Level Monthly KPI Aggregation

Collapse `loan_monthly_snapshot` into the numbers a risk management team 
reviews in their monthly meeting. Aggregation key: `month_number` (1–7), 
which maps cleanly to calendar months for the bulk of the portfolio.

| Output | Rows | Description |
|---|---|---|
| `portfolio_monthly_kpi.csv` | 7 | overall portfolio, one row per month |
| `portfolio_monthly_kpi_by_segment.csv` | ~1 000 | by `loan_type` and by `state` |

| KPI | Definition |
|---|---|
| `active_loans` | loans with a payment record in that month |
| `total_principal_outstanding` | sum of `principal` for active loans |
| `avg_on_time_rate` | mean `on_time_rate_cumul` across active loans |
| `median_on_time_rate` | median — more robust to outliers |
| `pct_delinquent` | share of loans with `on_time_rate_cumul < 1.0` |
| `pct_high_risk` | share of loans with `on_time_rate_cumul < 0.6` |
| `deteriorating_count` | loans where `is_deteriorating = 1` this month |
| `deteriorating_pct` | `deteriorating_count / active_loans` |

### 2.1 Normalize segment dimensions and derive due-date labels

In [ ]:
snap = loan_monthly_snapshot.copy()
snap["calendar_month"] = pd.to_datetime(snap["payment_due_date"], errors="coerce").dt.to_period("M").astype("string")
snap["loan_type"] = snap["loan_type"].str.strip().str.lower().replace("", pd.NA).fillna("unknown")
snap["state"] = snap["state"].str.strip().str.upper().replace("", pd.NA).fillna("Unknown")
snap["original_principal"] = snap["principal"].where(snap["principal"].ge(0)).round(2)

### 2.2 Add binary helper columns for vectorized aggregation

Using `groupby.agg()` with pre-computed binary columns is **~50× faster** than 
`groupby.apply(custom_fn)` on 640 K rows — same result, no timeout risk.

In [ ]:
known = snap["months_with_known_status"].gt(0)
snap["ever_delinquent"] = snap["months_delinquent_cumul"].gt(0).astype(float).where(known)
snap["high_risk"] = snap["on_time_rate_cumul"].lt(0.6).astype(float).where(known)

### 2.3 KPI aggregation helper

In [ ]:
def agg_kpis(frame, key):
    result = frame.groupby(key, sort=True).agg(
        active_loans=("loan_id", "size"),
        current_status_known_loans=("status_known", "sum"),
        historical_status_known_loans=("ever_delinquent", "count"),
        ever_delinquent_loans=("ever_delinquent", "sum"),
        high_risk_loans=("high_risk", "sum"),
        principal_known_loans=("original_principal", "count"),
        total_original_principal=("original_principal", lambda values: values.sum(min_count=1)),
        avg_on_time_rate=("on_time_rate_cumul", "mean"),
        median_on_time_rate=("on_time_rate_cumul", "median"),
        deterioration_eligible_loans=("is_deteriorating", "count"),
        deteriorating_count=("is_deteriorating", "sum"),
    )
    result["pct_delinquent"] = result["ever_delinquent_loans"] / result["historical_status_known_loans"].where(result["historical_status_known_loans"].gt(0))
    result["pct_high_risk"] = result["high_risk_loans"] / result["historical_status_known_loans"].where(result["historical_status_known_loans"].gt(0))
    result["deteriorating_pct"] = result["deteriorating_count"] / result["deterioration_eligible_loans"].where(result["deterioration_eligible_loans"].gt(0))

    count_cols = ["ever_delinquent_loans", "high_risk_loans", "deteriorating_count"]
    result[count_cols] = result[count_cols].astype(int)
    result["total_original_principal"] = result["total_original_principal"].round(2)
    rate_cols = ["avg_on_time_rate", "median_on_time_rate", "pct_delinquent", "pct_high_risk", "deteriorating_pct"]
    result[rate_cols] = result[rate_cols].astype(float).round(4)
    return result.reset_index()

### 2.4 Overall portfolio KPI — one row per month

In [ ]:
portfolio_monthly_kpi = agg_kpis(snap, "month_number")
representative_month = (
    snap.dropna(subset=["calendar_month"]).groupby("month_number")["calendar_month"]
    .agg(lambda dates: dates.mode().iloc[0])
    .rename("representative_month")
)
portfolio_monthly_kpi.insert(
    1, "representative_month", portfolio_monthly_kpi["month_number"].map(representative_month)
)
portfolio_monthly_kpi

### 2.5 Segment KPI — by `loan_type` × month

In [ ]:
kpi_by_loan_type = agg_kpis(snap, ["month_number", "loan_type"]).rename(columns={"loan_type": "segment_value"})
kpi_by_loan_type["segment_type"] = "loan_type"
kpi_by_loan_type["is_top10"] = pd.NA

### 2.6 Segment KPI — by `state` × month

All states are kept in the output (useful for choropleth maps). 
`is_top10` flags the states with the highest loan volume for quick-filter use.

In [ ]:
top10_states = (
    snap.loc[snap["state"].ne("Unknown")].groupby("state")["loan_id"]
    .size().nlargest(10).index
)
kpi_by_state = agg_kpis(snap, ["month_number", "state"]).rename(columns={"state": "segment_value"})
kpi_by_state["segment_type"] = "state"
kpi_by_state["is_top10"] = kpi_by_state["segment_value"].isin(top10_states).astype(int)

### 2.7 Combine → `portfolio_monthly_kpi_by_segment`

In [ ]:
kpi_cols = portfolio_monthly_kpi.columns.drop(["month_number", "representative_month"]).tolist()
seg_cols = ["month_number", "segment_type", "segment_value", "is_top10"] + kpi_cols
portfolio_monthly_kpi_by_segment = pd.concat([kpi_by_loan_type, kpi_by_state], ignore_index=True)[seg_cols]
portfolio_monthly_kpi_by_segment.head()

### 2.8 Sanity checks

In [ ]:
monthly = portfolio_monthly_kpi.set_index("month_number")
assert monthly["active_loans"].equals(reconciliation["source_loan_periods"])
reconciliation["reconciled_active_loans"] = monthly["active_loans"]

count_cols = [
    "active_loans", "current_status_known_loans", "historical_status_known_loans",
    "ever_delinquent_loans", "high_risk_loans", "principal_known_loans",
    "deterioration_eligible_loans", "deteriorating_count",
]
for dimension, segments in portfolio_monthly_kpi_by_segment.groupby("segment_type"):
    totals = segments.groupby("month_number")[count_cols].sum()
    pd.testing.assert_frame_equal(totals, monthly[count_cols], obj=f"{dimension} segment counts")
    principal = segments.groupby("month_number")["total_original_principal"].sum(min_count=1)
    pd.testing.assert_series_equal(principal.round(2), monthly["total_original_principal"], obj=f"{dimension} principal")
    reconciliation[f"{dimension}_segment_active_loans"] = totals["active_loans"]

for numerator, denominator in [
    ("ever_delinquent_loans", "historical_status_known_loans"),
    ("deteriorating_count", "deterioration_eligible_loans"),
    ("high_risk_loans", "ever_delinquent_loans"),
    ("current_status_known_loans", "historical_status_known_loans"),
    ("historical_status_known_loans", "active_loans"),
]:
    assert monthly[numerator].between(0, monthly[denominator]).all(), numerator
print("Source keys, monthly KPIs, and both segment dimensions reconciled.")

### 2.9 Export CSVs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
portfolio_monthly_kpi.to_csv(OUTPUT_DIR / "portfolio_monthly_kpi.csv", index=False)
portfolio_monthly_kpi_by_segment.to_csv(OUTPUT_DIR / "portfolio_monthly_kpi_by_segment.csv", index=False)
reconciliation.reset_index().to_csv(OUTPUT_DIR / "portfolio_kpi_reconciliation.csv", index=False)

---
## ✓ Step 2 Complete

| File | Rows | Ready for |
|---|---|---|
| `portfolio_monthly_kpi.csv` | 7 | overall trend lines, exec-level KPI card |
| `portfolio_monthly_kpi_by_segment.csv` | ~1 000 | loan_type bar charts, state choropleth |
| `loan_monthly_snapshot.csv` *(Step 1)* | 640 K | loan-level drill-down, sparklines |

**Next: Phase 1 — Dashboard** — these three files are the data layer.

In [ ]:
reconciliation